# 00 · 로드맵 — franka-depth-mppi로 MPC · MPPI · Isaac Sim 공부하기

[ukhyeon/franka-depth-mppi](https://github.com/ukhyeon/franka-depth-mppi)와 노션 「🎮 Issac sim」(U1–U6 학습 기록)을 단계별 노트북으로 다시 정리한 폴더다. 각 노트북은 아래 순서를 따른다.
1. 노션 기록 확인
2. 작은 토이 모델로 직접 구현
3. 레포·cuRobo 실제 코드 위치 확인
4. 실험 로그로 검증

## 전체 파이프라인

```text
D435i 깊이 (기록된 236 프레임 replay)
   │  hand_experiment.py:243-246   decay → add_camera_frame → ESDF → hash 갱신
   ▼
nvblox 지도 (occupancy, voxel 3 cm)  ──── 환경 충돌 비용 (η 25 mm, w 1e5)        ← 04
   │
   ├─ 손 ROI 관측점 → 1 mm 패치 ≤64개 ── 손 전용 충돌 비용 (η 25 / 73.52 mm)     ← 05
   ▼
cuRobo rollout: act_seq[400,30,7] → 관절 상태 → FK → 충돌 구 61개 → costs[400,30]  ← 03
   ▼
MPPI: sample → weight softmax(-S/β) → 평균·분산 갱신 ×5 → BEST 후보               ← 02
   ▼
MPC: 첫 행동만 실행, 매 physics step(1/60 s) 다시 풀기, warm start                 ← 01
   ▼
Isaac Sim Franka: 관절 위치 목표 → PD 드라이브 → 측정 상태 → 다시 MPC             ← 07
   ▼
로그(evidence/R·LOW·HIGH) → 이격·도달 시간·추종 분석                               ← 06
```

## 노트북 목록

| # | 노트북 | 노션 | 핵심 질문 | 필요 환경 |
|---|---|---|---|---|
| 00 | 로드맵 (이 노트) | — | 무엇을 어떤 순서로? | numpy |
| 01 | [U1 MPC](01_U1_mpc.ipynb) | U1 | 왜 첫 행동만 실행하고 다시 푸는가? 왜 장애물 앞에서 멈추는가? | numpy, scipy |
| 02 | [U2 MPPI](02_U2_mppi.ipynb) | U2 | 400 후보, β·γ·step size, BEST는 각각 무엇을 바꾸는가? | numpy, scipy |
| 03 | [U3 cuRobo rollout](03_U3_curobo_rollout.ipynb) | U3 | 가속도 후보가 어떻게 충돌 구와 비용이 되는가? η 비용식은? | **cuRobo + GPU** (일부 셀) |
| 04 | [U4 nvblox / ESDF](04_U4_nvblox_esdf.ipynb) | U4 | occupancy·TSDF·ESDF 차이, 미관측이 왜 비용 0이 되는가? | numpy, scipy |
| 05 | [U5 손 ROI 비용](05_U5_hand_roi_cost.ipynb) | U5 | 깊이 → ROI 패치 → 질량-η 정책 → optimizer 훅 | numpy (실제 depth 데이터) |
| 06 | [U6 실험 분석](06_U6_experiment_analysis.ipynb) | U6 | R/LOW/HIGH 이격·경로·도달·추종을 로그로 재계산 | numpy, matplotlib |
| 07 | [Isaac Sim MPC 실습](07_isaacsim_mpc_lab.ipynb) | 전체 | Isaac Sim 루프에서 η를 바꾸면? | **Isaac Sim + cuRobo** (실행 셀) |

**권장 순서:** 01 → 02 → 03은 이어지는 흐름이다(같은 점질량 장면 → Franka). 04 → 05 → 06은 프로젝트 고유 부분이다. 07은 언제든 손으로 돌려 보는 실습이다.

**공용 파일**
- `common.py`: 경로(`REPO`, `CUROBO`, `TAPE`, `EVIDENCE`), cuRobo 식을 numpy로 옮긴 함수(`dt_array`, `eta_cost`, `eta_policy`), `show_code`
- `toy.py`: 01·02용 2D 점질량, gradient MPC 비용, numpy MPPI
- `scripts/mpc_lab.py`: 07의 Isaac Sim 독립 실행 스크립트(결과는 `scripts/results/`)

모든 노트북에는 실행 결과와 그림이 미리 저장되어 있다. 다시 실행하지 않아도 읽을 수 있다.

## 실행 환경 (Jupyter 커널)

커널 이름은 **"Isaac Sim 4.5 + cuRobo"**(`isaacsim-curobo`)다. conda 가상환경 `isaacsim`(Python 3.13, numpy·torch 없음)과는 다른 환경이다. 이 커널은 실험(`scripts/replay.sh`)과 같은 **Isaac Sim 내장 파이썬 `~/isaacsim/python.sh`**(Python 3.10, torch 2.5.1+cu118, cuRobo)로 실행된다.

설정 내용 (2026-10-25 적용 완료):

| 구성 | 위치 | 역할 |
|---|---|---|
| ipykernel 등 | `study_franka_mppi/.kernel_pkgs/` | `pip install --target`로 따로 설치. Isaac Sim 내장 site-packages는 건드리지 않음 |
| 실행기 | `study_franka_mppi/kernel_launcher.py` | cuRobo 경로를 앞에, `.kernel_pkgs`를 sys.path **맨 뒤**에 붙임 → Isaac Sim에 이미 있는 패키지는 Isaac 버전 사용. 단 `typing_extensions`만 새 버전을 먼저 불러옴(jupyter_client 요구) |
| kernelspec | `~/.local/share/jupyter/kernels/isaacsim-curobo/kernel.json` | `python.sh kernel_launcher.py -f {connection_file}`, `PYTHONUTF8=1` |

그래서 Isaac Sim을 그냥 `python.sh`로 실행할 때(`replay.sh`, `mpc_lab.py`)는 아무 변화가 없다. 노트북 메타데이터에도 이 커널을 지정해 두어서, VSCode에서 열면 자동으로 선택된다. 목록에 안 보이면 커널 선택 → "Jupyter Kernel..." → "Isaac Sim 4.5 + cuRobo"를 고르고, 그래도 없으면 VSCode 창을 다시 연다.

다른 PC에서 다시 만들 때:

```bash
cd ~/isaacsim && ./python.sh -m pip install --target ~/isaacsim/study_franka_mppi/.kernel_pkgs ipykernel
mkdir -p ~/.local/share/jupyter/kernels/isaacsim-curobo
cat > ~/.local/share/jupyter/kernels/isaacsim-curobo/kernel.json <<'EOF'
{
  "argv": ["~/isaacsim/python.sh", "~/isaacsim/study_franka_mppi/kernel_launcher.py", "-f", "{connection_file}"],
  "display_name": "Isaac Sim 4.5 + cuRobo",
  "language": "python",
  "env": {"PYTHONUTF8": "1"},
  "metadata": {"debugger": false}
}
EOF
```

- cuRobo가 필요 없는 01·02·04·05·06은 numpy·scipy·matplotlib가 있는 다른 커널에서도 돈다.
- 07의 Isaac Sim 실행 셀은 노트북 커널 안에서 SimulationApp을 띄우지 않는다. `subprocess`로 `python.sh scripts/mpc_lab.py`를 호출한다.
- conda 환경이 켜진 셸에서 `python.sh`를 실행하면 경고가 뜬다(`CONDA_PREFIX`). 동작에는 문제없다.

In [1]:
from common import *
check_paths()
import importlib
for m in ['numpy', 'scipy', 'matplotlib', 'yaml', 'torch', 'curobo']:
    try:
        mod = importlib.import_module(m)
        print(f'{m:11s} OK  {getattr(mod, "__version__", "")}')
    except ImportError:
        print(f'{m:11s} 없음')

REPO      OK  ~/franka-depth-mppi
CUROBO    OK  ~/curobo
TAPE      OK  ~/franka-depth-mppi/data/hand-tape-20261012-000004-472steps-registered
EVIDENCE  OK  ~/franka-depth-mppi/evidence
numpy       OK  1.26.0
scipy       OK  1.15.3
matplotlib  OK  3.8.4
yaml        OK  6.0.1
torch       OK  2.5.1+cu118
curobo      OK  0.7.7.post1.dev0+dirty


## 노션 기록 대비 정정·보완된 사항 모음

노트북을 만들면서 레포 로그·cuRobo 소스로 다시 확인한 결과를 모았다. 노션을 업데이트할 때 참고할 목록이다.

| 주제 | 노션 기록 | 확인 결과 | 근거 노트북 |
|---|---|---|---|
| horizon 길이 | 30 step ≈ 0.75 s (dt 0.02) | 최종 replay는 `--model-dt 1/60` → **0.675 s** (로그 `trajectory_duration_s`) | 03 |
| MPC dt vs physics dt | 0.02 vs 0.0167, 비율 1.2 | 최종 실험은 둘 다 1/60 s | 03, 06 |
| MPC 입력 상태 | 직전 MPC 명령을 다음 입력으로 사용 | 최종 replay는 `--state-input measured`. 그래서 `input_actual_q_max_rad`는 정의상 0. 실제 차이는 state filter 출력−실제(최대 ≈0.036 rad)에서 생김 | 01, 06 |
| 후보 400개 구성 | 380 / 0 가속도 10 / −평균 10 | 코드로 확인 (`particle_opt_base.py:286-289`), 마지막 noise = 0 (`parallel_mppi.py:582`) | 02 |
| 분산 갱신 | $0.99\Sigma+0.01\Sigma_w+\kappa$ | 맞음. 편차는 **갱신 전 평균** 기준, DIAG_A는 축별·horizon 평균 | 02 |
| 충돌 구 수 | 61개 | 구 배열은 65칸, 활성(r>0) 61개. 로그 반지름 = 원래 반지름 + 0.02 | 03, 05 |
| η 비용식 | sphere–OBB 식 | 커널 `scale_eta_metric`에서 $x=\eta-c$로 같은 식 확인. 겹친 뒤 기울기는 η와 무관 | 03 |
| nvblox integrator | TSDF / ESDF 설명 | 이 실험은 `integrator_type='occupancy'`(voxel 0.03 m). TSDF는 개념 학습용 | 04 |
| 미관측 비용 0 | −100 sentinel → 비용 0 | 로그로 확인. 다만 sentinel→0 처리는 nvblox_torch 커널 쪽이라 cuRobo 파이썬에선 안 보임 | 04 |
| "같은 자세 ROI 평가 18,502.6 / 217,650.1" | 별도 평가 | R 실행 중 기록된 **진단값**(α=0으로 optimizer에 미연결) | 05, 06 |
| HIGH의 η 기록 | 73.52 mm | `cost-connection.json`에는 덮어쓰기 전 값 42.87 mm가 남음. 실제 값은 `mass-policy.json`·`eta-after-run.json` | 06 |
| feasible | ROI가 feasible에 포함되는지 확인 필요 | 포함 안 됨. 세 조건 모두 100% feasible(R의 겹침 스텝 포함) | 05, 06 |
| comparison.csv | — | 13개 항목을 로그에서 재계산해 세 조건 모두 일치 | 06 |
| 손 정면 통과 높이 | LOW 0.505 / HIGH p=1 0.627 m | 로그 재계산 0.506 / 0.626 m (p=0.5 로그는 레포에 없음) | 06 |

## 노션 U4–U6 체크리스트 진행 상황

| 항목 | 상태 |
|---|---|
| U4 깊이 → 지도 반영 경로, 실제 사용 layer | 04에서 확인 (occupancy → ESDF) |
| U4 nvblox CUDA 내부 η 비용식, 미관측 처리, decay 계수 | **미확인** — nvblox_torch 소스(`query_sphere_sdf_cost`)를 봐야 함 |
| U5 ROI 비용 삽입 위치·결합, 질량→η 변환 | 05에서 확인 |
| U5 feasible 포함 여부, 전 후보 침범 시 동작 | 02·05·06에서 확인 (미포함, 덜 나쁜 후보 선택) |
| U5 ROI 관측 상실 시 비활성화 | 05에서 코드 확인 (해당 갱신에서 enable=False) |
| U6 입력/실제 상태 차이, dt 관계, 이격·도달·추종 | 06에서 확인 |
| U6 여러 seed 반복 | **미수행** — 레포 evidence는 조건별 단일 실행 |

## 공부 팁

- 각 노트북 끝의 **복습 질문**은 답을 `<details>`로 접어 두었다. 먼저 답해 보고 펼쳐서 확인한다.
- 토이 셀의 파라미터(β, horizon, init_cov, η)를 바꿔 다시 실행해 보는 것이 가장 빠르게 감을 잡는 방법이다.
- 실제 코드 줄 번호는 로컬 `~/franka-depth-mppi`(GitHub HEAD `72e8bac`)와 `~/curobo`(0.7.7, `0a50de1`, `particle_opt_base.py`에 한글 주석이 있는 로컬본) 기준이다.